In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings

# NLP and Text Processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Scikit-learn for ML Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')

# Download required NLTK data
print("Downloading NLTK resources...")
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

In [ ]:
def load_data(filepath):
    """Load IMDb review data from CSV"""
    print("\n" + "="*70)
    print("STEP 1: LOADING DATA")
    print("="*70)
    
    try:
        df = pd.read_csv(filepath)
        print(f"✓ Data loaded successfully from {filepath}")
        print(f"  Dataset shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"\nFirst 3 rows:")
        print(df.head(3))
        print(f"\nSentiment distribution:")
        print(df['sentiment'].value_counts())
        print(f"\nMissing values:")
        print(df.isnull().sum())
        return df
    except FileNotFoundError:
        print(f"✗ File not found: {filepath}")
        print("\nCreating sample dataset for demonstration...")
        return create_sample_dataset()
        
def create_sample_dataset():
    """Create sample IMDb-like dataset for demonstration"""
    sample_reviews = [
        ("This movie was absolutely fantastic! I loved every minute of it.", "positive"),
        ("Terrible film, waste of time and money.", "negative"),
        ("Great acting and wonderful cinematography throughout.", "positive"),
        ("Boring plot and poor dialogue, very disappointing.", "negative"),
        ("Masterpiece! One of the best movies I've ever seen.", "positive"),
        ("Awful movie, couldn't even finish watching it.", "negative"),
        ("Outstanding performances by the entire cast.", "positive"),
        ("Horrible screenplay and terrible direction.", "negative"),
        ("Excellent story with amazing visual effects.", "positive"),
        ("Dull, uninspiring, and completely forgettable.", "negative"),
        ("Best movie of the year, highly recommend!", "positive"),
        ("Unwatchable garbage, absolutely terrible.", "negative"),
    ]
    
    df = pd.DataFrame(sample_reviews, columns=['review', 'sentiment'])
    print(f"✓ Sample dataset created: {df.shape}")
    return df


In [ ]:
class TextPreprocessor:
    """Handles all text cleaning and preprocessing operations"""
    
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.punct = set(string.punctuation)
    
    def clean_text(self, text):
        """
        Clean a single text string:
        - Convert to lowercase
        - Remove special characters and extra whitespace
        - Remove punctuation
        """
        if not isinstance(text, str):
            return ""
        
        # Lowercase
        text = text.lower()
        
        # Remove URLs
        text = self.remove_urls(text)
        
        # Remove special characters but keep spaces
        text = ''.join(char if char.isalnum() or char.isspace() else ' ' for char in text)
        
        # Remove extra whitespace
        text = ' '.join(text.split())
        
        return text
    
    def remove_urls(self, text):
        """Remove URLs from text"""
        import re
        return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    def remove_stopwords(self, text):
        """Remove common English stopwords"""
        tokens = text.split()
        return ' '.join([word for word in tokens if word not in self.stop_words])
    
    def preprocess_corpus(self, texts):
        """Apply full preprocessing pipeline to corpus"""
        print("\n" + "="*70)
        print("STEP 2: TEXT PREPROCESSING")
        print("="*70)
        
        print("  Cleaning text (lowercasing, removing URLs, special characters)...")
        cleaned = [self.clean_text(text) for text in texts]
        
        print("  Removing stopwords...")
        processed = [self.remove_stopwords(text) for text in cleaned]
        
        print(f"✓ Preprocessing complete for {len(processed)} reviews")
        print(f"\nExample of processed text:")
        print(f"  Original: {texts[0][:100]}...")
        print(f"  Processed: {processed[0][:100]}...")
        
        return processed


In [ ]:
def extract_features(X_train, X_test, method='tfidf', max_features=5000):
    """
    Convert text to numerical features using TF-IDF or CountVectorizer
    
    Args:
        X_train: Training text data
        X_test: Testing text data
        method: 'tfidf' or 'count'
        max_features: Maximum number of features to extract
    
    Returns:
        X_train_vec, X_test_vec, vectorizer
    """
    print("\n" + "="*70)
    print("STEP 3: FEATURE EXTRACTION")
    print("="*70)
    
    if method == 'tfidf':
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            min_df=2,  # Ignore terms that appear in only one document
            max_df=0.95,  # Ignore terms that appear in >95% of documents
            ngram_range=(1, 2),  # Use unigrams and bigrams
            sublinear_tf=True
        )
        print(f"  Using TF-IDF Vectorizer")
    else:
        vectorizer = CountVectorizer(
            max_features=max_features,
            min_df=2,
            max_df=0.95,
            ngram_range=(1, 2)
        )
        print(f"  Using Count Vectorizer")
    
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    print(f"  Training feature matrix shape: {X_train_vec.shape}")
    print(f"  Testing feature matrix shape: {X_test_vec.shape}")
    print(f"  Total features extracted: {len(vectorizer.get_feature_names_out())}")
    print(f"✓ Feature extraction complete")
    
    return X_train_vec, X_test_vec, vectorizer

In [ ]:
def train_models(X_train, X_test, y_train, y_test):
    """
    Train Naive Bayes and Logistic Regression classifiers
    
    Returns:
        dict with trained models and their predictions
    """
    print("\n" + "="*70)
    print("STEP 4: MODEL TRAINING")
    print("="*70)
    
    results = {}
    
    # Naive Bayes
    print("\n  Training Multinomial Naive Bayes...")
    nb_model = MultinomialNB(alpha=1.0)
    nb_model.fit(X_train, y_train)
    nb_pred = nb_model.predict(X_test)
    nb_pred_proba = nb_model.predict_proba(X_test)[:, 1]
    
    results['Naive Bayes'] = {
        'model': nb_model,
        'predictions': nb_pred,
        'probabilities': nb_pred_proba
    }
    print("  ✓ Naive Bayes training complete")
    
    # Logistic Regression (Baseline)
    print("  Training Logistic Regression (Baseline)...")
    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(X_train, y_train)
    lr_pred = lr_model.predict(X_test)
    lr_pred_proba = lr_model.predict_proba(X_test)[:, 1]
    
    results['Logistic Regression'] = {
        'model': lr_model,
        'predictions': lr_pred,
        'probabilities': lr_pred_proba
    }
    print("  ✓ Logistic Regression training complete")
    
    return results

def evaluate_models(results, y_test):
    """
    Evaluate and compare all trained models
    
    Returns:
        DataFrame with evaluation metrics
    """
    print("\n" + "="*70)
    print("STEP 5: MODEL EVALUATION")
    print("="*70)
    
    metrics_list = []
    
    for model_name, model_data in results.items():
        y_pred = model_data['predictions']
        y_proba = model_data['probabilities']
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        metrics_list.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC-AUC': roc_auc
        })
        
        print(f"\n{model_name}:")
        print(f"  Accuracy:  {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
        print(f"  ROC-AUC:   {roc_auc:.4f}")
        
        print(f"\nClassification Report:")
        print(classification_report(y_test, y_pred, 
                                   target_names=['Negative', 'Positive']))
    
    return pd.DataFrame(metrics_list)

In [ ]:
def extract_important_words(model, vectorizer, top_n=15):
    """
    Extract most important words for each sentiment class
    
    Returns:
        dict with positive and negative words
    """
    feature_names = np.array(vectorizer.get_feature_names_out())
    
    # Get feature importance (log probabilities for Naive Bayes)
    if hasattr(model, 'feature_log_prob_'):
        # Naive Bayes
        pos_class_idx = 1
        neg_class_idx = 0
        
        pos_scores = model.feature_log_prob_[pos_class_idx]
        neg_scores = model.feature_log_prob_[neg_class_idx]
    else:
        # Logistic Regression
        pos_scores = model.coef_[0]
        neg_scores = -model.coef_[0]
    
    # Get top positive and negative words
    top_pos_indices = pos_scores.argsort()[-top_n:][::-1]
    top_neg_indices = neg_scores.argsort()[-top_n:][::-1]
    
    positive_words = {
        'words': feature_names[top_pos_indices].tolist(),
        'scores': pos_scores[top_pos_indices].tolist()
    }
    
    negative_words = {
        'words': feature_names[top_neg_indices].tolist(),
        'scores': neg_scores[top_neg_indices].tolist()
    }
    
    return positive_words, negative_words

def analyze_word_patterns(df, preprocessor):
    """
    Analyze most common words in positive and negative reviews
    """
    print("\n" + "="*70)
    print("STEP 6: WORD PATTERN ANALYSIS")
    print("="*70)
    
    # Preprocess reviews
    df['processed_review'] = df['review'].apply(
        lambda x: preprocessor.remove_stopwords(preprocessor.clean_text(x))
    )
    
    # Analyze positive reviews
    positive_reviews = df[df['sentiment'] == 'positive']['processed_review']
    positive_text = ' '.join(positive_reviews)
    positive_words = Counter(positive_text.split())
    
    # Analyze negative reviews
    negative_reviews = df[df['sentiment'] == 'negative']['processed_review']
    negative_text = ' '.join(negative_reviews)
    negative_words = Counter(negative_text.split())
    
    return positive_words, negative_words

In [ ]:
def plot_model_comparison(metrics_df):
    """Compare model performance across metrics"""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    metrics_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
        kind='bar', ax=ax, width=0.8
    )
    
    ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    return fig

def plot_important_words(model, vectorizer, model_name='Naive Bayes', top_n=15):
    """Visualize most important words for each sentiment class"""
    positive_words, negative_words = extract_important_words(model, vectorizer, top_n)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Positive words
    ax1.barh(range(len(positive_words['words'])), positive_words['scores'], color='green', alpha=0.7)
    ax1.set_yticks(range(len(positive_words['words'])))
    ax1.set_yticklabels(positive_words['words'], fontsize=10)
    ax1.set_xlabel('Importance Score', fontsize=11)
    ax1.set_title(f'Top {top_n} Words Predictive of POSITIVE Sentiment\n({model_name})', 
                  fontsize=12, fontweight='bold')
    ax1.invert_yaxis()
    ax1.grid(axis='x', alpha=0.3)
    
    # Negative words
    ax2.barh(range(len(negative_words['words'])), negative_words['scores'], color='red', alpha=0.7)
    ax2.set_yticks(range(len(negative_words['words'])))
    ax2.set_yticklabels(negative_words['words'], fontsize=10)
    ax2.set_xlabel('Importance Score', fontsize=11)
    ax2.set_title(f'Top {top_n} Words Predictive of NEGATIVE Sentiment\n({model_name})', 
                  fontsize=12, fontweight='bold')
    ax2.invert_yaxis()
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_word_frequency(positive_words, negative_words, top_n=15):
    """Visualize most frequent words in positive and negative reviews"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Positive words frequency
    pos_word_freq = dict(positive_words.most_common(top_n))
    ax1.barh(list(pos_word_freq.keys()), list(pos_word_freq.values()), color='steelblue', alpha=0.7)
    ax1.set_xlabel('Frequency', fontsize=11)
    ax1.set_title(f'Top {top_n} Most Frequent Words in POSITIVE Reviews', 
                  fontsize=12, fontweight='bold')
    ax1.invert_yaxis()
    ax1.grid(axis='x', alpha=0.3)
    
    # Negative words frequency
    neg_word_freq = dict(negative_words.most_common(top_n))
    ax2.barh(list(neg_word_freq.keys()), list(neg_word_freq.values()), color='coral', alpha=0.7)
    ax2.set_xlabel('Frequency', fontsize=11)
    ax2.set_title(f'Top {top_n} Most Frequent Words in NEGATIVE Reviews', 
                  fontsize=12, fontweight='bold')
    ax2.invert_yaxis()
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_confusion_matrices(results, y_test):
    """Plot confusion matrices for all models"""
    fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5))
    
    if len(results) == 1:
        axes = [axes]
    
    for idx, (model_name, model_data) in enumerate(results.items()):
        cm = confusion_matrix(y_test, model_data['predictions'])
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                   xticklabels=['Negative', 'Positive'],
                   yticklabels=['Negative', 'Positive'],
                   cbar=False)
        axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')
        axes[idx].set_ylabel('True Label', fontsize=11)
        axes[idx].set_xlabel('Predicted Label', fontsize=11)
    
    plt.tight_layout()
    return fig

def plot_sentiment_distribution(df):
    """Plot distribution of sentiments in dataset"""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    sentiment_counts = df['sentiment'].value_counts()
    colors = ['green' if label == 'positive' else 'red' for label in sentiment_counts.index]
    
    sentiment_counts.plot(kind='bar', ax=ax, color=colors, alpha=0.7)
    ax.set_title('Sentiment Distribution in Dataset', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count', fontsize=12)
    ax.set_xlabel('Sentiment', fontsize=12)
    ax.set_xticklabels(sentiment_counts.index, rotation=0)
    ax.grid(axis='y', alpha=0.3)
    
    # Add count labels on bars
    for i, v in enumerate(sentiment_counts):
        ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    
    plt.tight_layout()
    return fig

In [ ]:
def main(data_path=None):
    """
    Execute the complete sentiment analysis pipeline
    
    Args:
        data_path: Path to CSV file with 'sentiment' and 'review' columns
    """
    print("\n" + "="*70)
    print("IMDb MOVIE REVIEW SENTIMENT ANALYSIS PIPELINE")
    print("="*70)
    
    # 1. Load Data
    if data_path is None:
        # Try to load from common locations or create sample
        data_path = 'imdb_reviews.csv'
    
    df = load_data(data_path)
    
    # 2. Preprocess Text
    preprocessor = TextPreprocessor()
    df['processed_review'] = df['review'].apply(preprocessor.preprocess_corpus)
    # Convert back to Series of strings for sklearn
    df['processed_review'] = df['processed_review'].apply(lambda x: x[0] if isinstance(x, list) else x)
    
    # Apply preprocessing row by row
    processed_reviews = []
    for review in df['review']:
        cleaned = preprocessor.clean_text(review)
        processed = preprocessor.remove_stopwords(cleaned)
        processed_reviews.append(processed)
    df['processed_review'] = processed_reviews
    
    # 3. Train-Test Split
    print("\n" + "="*70)
    print("STEP 3: TRAIN-TEST SPLIT")
    print("="*70)
    
    X = df['processed_review']
    y = (df['sentiment'] == 'positive').astype(int)  # Convert to binary (0=negative, 1=positive)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"  Training set size: {len(X_train)}")
    print(f"  Testing set size: {len(X_test)}")
    print(f"  Training set positive ratio: {y_train.mean():.2%}")
    print(f"  Testing set positive ratio: {y_test.mean():.2%}")
    
    # 4. Feature Extraction
    X_train_vec, X_test_vec, vectorizer = extract_features(X_train, X_test, method='tfidf')
    
    # 5. Train Models
    results = train_models(X_train_vec, X_test_vec, y_train, y_test)
    
    # 6. Evaluate Models
    metrics_df = evaluate_models(results, y_test)
    
    # 7. Analyze Words
    positive_word_freq, negative_word_freq = analyze_word_patterns(df, preprocessor)
    
    print(f"\nMost common words in positive reviews:")
    for word, count in positive_word_freq.most_common(10):
        print(f"  {word}: {count}")
    
    print(f"\nMost common words in negative reviews:")
    for word, count in negative_word_freq.most_common(10):
        print(f"  {word}: {count}")
    
    # 8. Generate Visualizations
    print("\n" + "="*70)
    print("STEP 7: GENERATING VISUALIZATIONS")
    print("="*70)
    
    figures = {}
    
    # Sentiment distribution
    fig1 = plot_sentiment_distribution(df)
    figures['sentiment_distribution'] = fig1
    print("  ✓ Sentiment distribution plot created")
    
    # Model comparison
    fig2 = plot_model_comparison(metrics_df)
    figures['model_comparison'] = fig2
    print("  ✓ Model comparison plot created")
    
    # Important words - Naive Bayes
    nb_model = results['Naive Bayes']['model']
    fig3 = plot_important_words(nb_model, vectorizer, model_name='Naive Bayes', top_n=15)
    figures['important_words_nb'] = fig3
    print("  ✓ Important words plot (Naive Bayes) created")
    
    # Important words - Logistic Regression
    lr_model = results['Logistic Regression']['model']
    fig4 = plot_important_words(lr_model, vectorizer, model_name='Logistic Regression', top_n=15)
    figures['important_words_lr'] = fig4
    print("  ✓ Important words plot (Logistic Regression) created")
    
    # Word frequency
    fig5 = plot_word_frequency(positive_word_freq, negative_word_freq, top_n=15)
    figures['word_frequency'] = fig5
    print("  ✓ Word frequency plot created")
    
    # Confusion matrices
    fig6 = plot_confusion_matrices(results, y_test)
    figures['confusion_matrices'] = fig6
    print("  ✓ Confusion matrices plot created")
    
    # Save all figures
    print("\n  Saving visualizations to PNG files...")
    for name, fig in figures.items():
        filepath = f'{name}.png'
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"    ✓ {filepath}")
    
    plt.close('all')
    
    # 9. Summary Report
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    print(f"\nDataset Statistics:")
    print(f"  Total reviews: {len(df)}")
    print(f"  Positive reviews: {(df['sentiment'] == 'positive').sum()} ({(df['sentiment'] == 'positive').mean():.1%})")
    print(f"  Negative reviews: {(df['sentiment'] == 'negative').sum()} ({(df['sentiment'] == 'negative').mean():.1%})")
    
    print(f"\nModel Performance Summary:")
    print(metrics_df.to_string(index=False))
    
    print(f"\nBest Model: {metrics_df.loc[metrics_df['F1-Score'].idxmax(), 'Model']}")
    print(f"  F1-Score: {metrics_df['F1-Score'].max():.4f}")
    
    print("\n" + "="*70)
    print("Pipeline execution completed successfully!")
    print("="*70 + "\n")
    
    return df, results, metrics_df, vectorizer, figures

In [ ]:
if __name__ == "__main__":
    # Run the pipeline
    # If you have the CSV file, pass the path:
    # df, results, metrics, vectorizer, figs = main('path_to_your_file.csv')
    
    # Otherwise, it will create a sample dataset:
    df, results, metrics, vectorizer, figures = main()